<a href="https://colab.research.google.com/github/andresmqz/crypto-news-classifier/blob/main/crypto_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

⚠️ Don't use Run all. Sections 9–11 overwrite the train/test split and call the Claude API (costs money). Run cells one by one.

## 1 · Connect Drive and set file paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DIR = "/content/drive/MyDrive/crypto-classifier"
os.makedirs(DIR, exist_ok=True)
POOL = f"{DIR}/pool.csv"        # everything collected
LABELED = f"{DIR}/labeled.csv"  # your labels

import pandas as pd

Mounted at /content/drive


## 6 · Finding dupes.

In [2]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer, util

done = pd.read_csv(LABELED)
news = done[done.label != "skip"].reset_index(drop=True)

model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(news.headline.tolist(), convert_to_tensor=True)
sim = util.cos_sim(emb, emb)

pairs = []
for i in range(len(news)):
    for j in range(i + 1, len(news)):
        pairs.append((sim[i][j].item(), i, j))
pairs.sort(reverse=True)

for score, i, j in pairs[:40]:
    warn = "" if news.label[i] == news.label[j] else "  ⚠️ different labels"
    print(f"{score:.2f}{warn}\n   {news.headline[i]}\n   {news.headline[j]}\n")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.99
   On-chain brokerage Spicenet raises $5M Series A led by dao5
   On-chain brokerage network Spicenet raises $5M Series A led by dao5

0.99
   Binance loses over $6 billion in 'biggest crypto hack in history'
   Binance loses over $6 billion in the largest crypto hack to date

0.98
   Standard Chartered signs definitive deal to acquire Zodia Custody
   Standard Chartered signs definitive agreements to acquire Zodia Custody

0.98
   1win’s $1WIN Token Goes Live After 15x Oversubscribed Public Sale
   1win’s $1WIN Token Goes Live After 15x Oversubscribed Sale

0.97
   Fireblocks denies report of $2.5 billion acquisition talks with Visa
   Fireblocks Denies Report of $2.5B Visa Acquisition Talks

0.96
   Liquid Network Restores Full Bitcoin Backing Two Months After $320M Exploit
   Liquid Network restores full bitcoin backing following $320 million exploit

0.96
   Justin Sun seeks sanctions against Trump-linked World Liberty Financial over UAE ties
   Justin Sun seeks sanctions agai

In [3]:
for low in [0.80, 0.75, 0.70, 0.65, 0.60]:
    band = [p for p in pairs if low <= p[0] < low + 0.05][:4]
    print(f"===== {low:.2f} – {low + 0.05:.2f} =====\n")
    for score, i, j in band:
        warn = "" if news.label[i] == news.label[j] else "  ⚠️ different labels"
        print(f"{score:.2f}{warn}\n   {news.headline[i]}\n   {news.headline[j]}\n")

===== 0.80 – 0.85 =====

0.85
   JPMorgan, Coca-Cola Among 7 More Companies Objecting to OKX-ICE Tokenized Stocks
   JPMorgan, Coca-Cola, Dell also say 'no' to OKX-ICE tokenized stock trading plan

0.85
   Uranium Digital raises $16.5 million Series A led by Blockchain Capital
   Uranium Digital raises $16.5 million to bring uranium trading on-chain

0.85
   CFTC clears stock perpetual futures for trading on US exchanges
   CFTC approves individual stock perpetual futures trading in US for the first time

0.85
   South Korea lawmakers mull pushing crypto tax implementation to 2029
   South Korea’s ruling party formally proposes deferral of crypto income tax

===== 0.75 – 0.80 =====

0.80
   1win Launches Two-Week Public Sale for $1WIN Token
   1win’s $1WIN Token Goes Live After 15x Oversubscribed Public Sale

0.80
   Standard Chartered Signs Definitive Deal To Acquire 100% Of Zodia Custody Limited: 16 outlets compared
   Standard Chartered to acquire Zodia, expanding crypto custody bus

In [4]:
THRESHOLD = 0.75

group = list(range(len(news)))
def find(x):
    while group[x] != x:
        x = group[x]
    return x

for score, i, j in pairs:
    if score < THRESHOLD:
        break
    group[find(i)] = find(j)

news["story"] = [find(i) for i in range(len(news))]
news["keep"] = ~news.story.duplicated()

print(f"{len(news)} usable headlines → {news.keep.sum()} distinct stories\n")
print(news[news.keep].label.value_counts())

print("\nBiggest groups:")
for s in news.story.value_counts().head(5).index:
    copies = news[news.story == s]
    print(f"  {len(copies)}× {copies.headline.iloc[0]}")

news[["headline", "label", "story", "keep"]].to_csv(f"{DIR}/stories.csv", index=False)

942 usable headlines → 686 distinct stories

label
price movement or market trading                    144
government regulation, taxation, or legal action    112
hack, exploit or security breach                    105
new product or feature launch                       102
institutional adoption or major industry shift       68
partnership or collaboration                         55
funding or investment                                52
acquisition or merger                                24
new token or coin launch                             24
Name: count, dtype: int64

Biggest groups:
  55× Binance arranges $6.34 billion to cover losses from record crypto hack
  13× Crypto custody specialist Fireblocks denies it's in talks to be acquired by Visa
  9× Strategy buys another $22 million of Bitcoin, increases planned share buybacks
  8× South Korea’s crypto bill nears completion while tax debate intensifies: reports
  7× Wells Fargo and five others object to OKX-ICE tokenized stock l

## 7 · Original 21

In [9]:
test21 = pd.DataFrame([
    ("France's parliament approves vote to hit stablecoin trades with 31% capital gains tax", "government regulation, taxation, or legal action"),
    ("Equity weakness could damp demand for bitcoin after biggest ETF inflows in a month", "price movement or market trading"),
    ("Bitcoin drops to $81,400 as Asian chip stocks tumble, options traders seek downside protection", "price movement or market trading"),
    ("South Korean IPO slump adds to AI market jitters as Brent crude tops $111", "price movement or market trading"),
    ("Bitcoin ETFs just saw their biggest day in a month thanks entirely to BlackRock", "price movement or market trading"),
    ("Cardano founder Charles Hoskinson is building a new programming language for the network", "new product or feature launch"),
    ("Goldman, Wells Fargo, Morgan Stanley see tokenization as a growth play, not a threat", "institutional adoption or major industry shift"),
    ("Deutsche Bank targets Asia crypto custody expansion in 2027 as German launch nears", "new product or feature launch"),
    ("SEC tokenized stock framework faces issuer pushback as Broadcom, Verizon object to OKX-ICE listing", "government regulation, taxation, or legal action"),
    ("Latvia eyes crypto tax payments in 2027 as parliament overhauls crypto taxation", "government regulation, taxation, or legal action"),
    ("Polygon sees Coinme acquisition as entry to payments sector dominated by Visa, Stripe", "acquisition or merger"),
    ("JPMorgan expands 'JPM Coin' deposit token to Ethereum for institutional clients", "new product or feature launch"),
    ("Farcaster acquired by Balaji's Network School, with new 'social smart contract' for users", "acquisition or merger"),
    ("Helius CEO Mert Mumtaz joins NEAR Foundation Council", "partnership or collaboration"),
    ("StarkWare launches OffMarket to let Polymarket users trade more privately", "new product or feature launch"),
    ("Swiss regulator FINMA approves country's first spot bitcoin ETF for retail investors", "government regulation, taxation, or legal action"),
    ("Cysic suspends bridge after attacker mints tokens and moves $727,000 to Ethereum", "hack, exploit or security breach"),
    ("Magic Eden legacy approvals leave $5.7 million in NFTs exposed to exploit before rescue", "hack, exploit or security breach"),
    ("Bitget starts phased withdrawal resumption following $388 million exploit", "hack, exploit or security breach"),
    ("Securitize launches 1:1-backed tokenized stocks, starting with Apple, Nvidia, Strategy and more", "new product or feature launch"),
    ("Squid launches native token QUID, plans public sale to start on June 30", "new token or coin launch"),
], columns=["headline", "label"])
test21.to_csv(f"{DIR}/test21.csv", index=False)

done = pd.read_csv(LABELED)
emb_test = model.encode(test21.headline.tolist(), convert_to_tensor=True)
emb_all = model.encode(done.headline.tolist(), convert_to_tensor=True)
sim21 = util.cos_sim(emb_test, emb_all)

overlap = []
for t in range(len(test21)):
    for d in range(len(done)):
        score = sim21[t][d].item()
        if score >= 0.65:
            overlap.append((score, test21.headline[t], done.headline[d]))
overlap.sort(reverse=True)

for score, t, d in overlap:
    flag = "LEAK" if score >= 0.75 else "check"
    print(f"{score:.2f} [{flag}]\n   test:    {t}\n   dataset: {d}\n")

leaks = pd.DataFrame([o for o in overlap if o[0] >= 0.75], columns=["score", "test_headline", "headline"])
leaks.to_csv(f"{DIR}/test21_overlap.csv", index=False)
print(f"{leaks.headline.nunique()} dataset headlines overlap with the original 21")

0.86 [LEAK]
   test:    Bitcoin ETFs just saw their biggest day in a month thanks entirely to BlackRock
   dataset: Bitcoin ETFs Post Their Best Day in Nearly a Month as BlackRock Does All the Buying

0.84 [LEAK]
   test:    Polygon sees Coinme acquisition as entry to payments sector dominated by Visa, Stripe
   dataset: Polygon Labs completes Coinme acquisition to challenge Visa and Stripe in global payments

0.82 [LEAK]
   test:    Swiss regulator FINMA approves country's first spot bitcoin ETF for retail investors
   dataset: Swiss Regulator Approves First Retail Fund That Holds Bitcoin Directly

0.79 [LEAK]
   test:    Polygon sees Coinme acquisition as entry to payments sector dominated by Visa, Stripe
   dataset: Polygon Takes Aim At Visa And Stripe After Closing Its Coinme Purchase

0.76 [LEAK]
   test:    Magic Eden legacy approvals leave $5.7 million in NFTs exposed to exploit before rescue
   dataset: Magic Eden Exploit: $2.8M Stolen, $5.7M in NFTs Saved [2026]

0.73 [check]


In [10]:
extra = pd.DataFrame([
    (0.73, "JPMorgan expands 'JPM Coin' deposit token to Ethereum for institutional clients",
     "JPMorgan's JPM Coin Arrives on Ethereum, Putting Bank Dollars on a Public Blockchain"),
    (0.71, "France's parliament approves vote to hit stablecoin trades with 31% capital gains tax",
     "French Lawmakers Back Stablecoin Swap Tax by Just One Vote"),
], columns=["score", "test_headline", "headline"])

leaks = pd.concat([leaks, extra])
leaks.to_csv(f"{DIR}/test21_overlap.csv", index=False)
print(f"{len(leaks)} dataset headlines set aside")

7 dataset headlines set aside


## 8 · First classifier

In [ ]:
from sklearn.linear_model import LogisticRegression

leaks = pd.read_csv(f"{DIR}/test21_overlap.csv")
train = news[news.keep & ~news.headline.isin(leaks.headline)].reset_index(drop=True)
print(f"Training on {len(train)} distinct stories\n")

X = model.encode(train.headline.tolist())
y = train.label

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X, y)

test21 = pd.read_csv(f"{DIR}/test21.csv")
probs = clf.predict_proba(model.encode(test21.headline.tolist()))
ranked = probs.argsort(axis=1)[:, ::-1]

top1 = top2 = 0
for k, row in test21.iterrows():
    first, second = clf.classes_[ranked[k, 0]], clf.classes_[ranked[k, 1]]
    top1 += first == row.label
    top2 += row.label in (first, second)
    mark = "✅" if first == row.label else ("🟡" if second == row.label else "❌")
    print(f"{mark} 1st: {first} | 2nd: {second} | you: {row.label} | {row.headline}")

print(f"\nYour classifier — Top-1: {top1}/21   Top-2: {top2}/21")
print("BART:  12/21 · Claude: 20/21")

Training on 252 distinct stories

✅ 1st: government regulation, taxation, or legal action | 2nd: price movement or market trading | you: government regulation, taxation, or legal action | France's parliament approves vote to hit stablecoin trades with 31% capital gains tax
✅ 1st: price movement or market trading | 2nd: institutional adoption or major industry shift | you: price movement or market trading | Equity weakness could damp demand for bitcoin after biggest ETF inflows in a month
✅ 1st: price movement or market trading | 2nd: institutional adoption or major industry shift | you: price movement or market trading | Bitcoin drops to $81,400 as Asian chip stocks tumble, options traders seek downside protection
✅ 1st: price movement or market trading | 2nd: new product or feature launch | you: price movement or market trading | South Korean IPO slump adds to AI market jitters as Brent crude tops $111
✅ 1st: price movement or market trading | 2nd: institutional adoption or major indu

## 9 · Build dataset

In [ ]:
done = pd.read_csv(LABELED)
pool = pd.read_csv(POOL)
leaks = pd.read_csv(f"{DIR}/test21_overlap.csv")

# a date for every headline: publish date, or collection day if missing
dates = pool[["headline", "collected"]]
done = done.merge(dates, on="headline", how="left")
done["date"] = pd.to_datetime(done.published, utc=True, errors="coerce", format="mixed")
done["date"] = done.date.fillna(pd.to_datetime(done.collected, utc=True))

# news: one headline per story, dated by the story's earliest headline
news_d = news.merge(done[["headline", "date"]], on="headline", how="left")
news_d = news_d[~news_d.headline.isin(leaks.headline)]
news_d["date"] = news_d.groupby("story").date.transform("min")
news_one = news_d[~news_d.story.duplicated()][["headline", "label", "date"]]

# noise: skips renamed, capped at 70
noise = done[done.label == "skip"].sample(n=70, random_state=42)[["headline", "date"]]
noise["label"] = "noise"

dataset = pd.concat([news_one, noise]).sort_values("date").reset_index(drop=True)
dataset.to_csv(f"{DIR}/dataset.csv", index=False)

print(f"{len(dataset)} examples, {dataset.date.isna().sum()} without a date\n")
print(dataset.label.value_counts())
print(f"\nOldest: {dataset.date.min()}\nNewest: {dataset.date.max()}")

431 examples, 0 without a date

label
noise                                               70
price movement or market trading                    67
government regulation, taxation, or legal action    56
new product or feature launch                       49
institutional adoption or major industry shift      48
partnership or collaboration                        41
hack, exploit or security breach                    38
funding or investment                               28
acquisition or merger                               18
new token or coin launch                            16
Name: count, dtype: int64

Oldest: 2025-12-29 16:10:19+00:00
Newest: 2026-10-28 15:00:02+00:00


In [ ]:
days = dataset.date.dt.date.value_counts().sort_index()
print("Before Oct 15:", days[days.index < pd.Timestamp("2026-10-15").date()].sum())
print()
print(days[days.index >= pd.Timestamp("2026-10-15").date()])

Before Oct 15: 10

date
2026-10-15     5
2026-10-16    11
2026-10-17     7
2026-10-18     3
2026-10-19    23
2026-10-20    30
2026-10-21    62
2026-10-22    66
2026-10-23    48
2026-10-24    18
2026-10-25     6
2026-10-26    49
2026-10-27    55
2026-10-28    38
Name: count, dtype: int64


In [ ]:
CUTOFF = pd.Timestamp("2026-10-27", tz="UTC")

train = dataset[dataset.date < CUTOFF].reset_index(drop=True)
test = dataset[dataset.date >= CUTOFF].reset_index(drop=True)

train.to_csv(f"{DIR}/train.csv", index=False)
test.to_csv(f"{DIR}/test.csv", index=False)

print(f"Train: {len(train)}   Test: {len(test)}\n")
print(pd.DataFrame({"train": train.label.value_counts(),
                    "test": test.label.value_counts()}).fillna(0).astype(int))

Train: 338   Test: 93

                                                  train  test
label                                                        
acquisition or merger                                16     2
funding or investment                                20     8
government regulation, taxation, or legal action     46    10
hack, exploit or security breach                     31     7
institutional adoption or major industry shift       40     8
new product or feature launch                        32    17
new token or coin launch                             12     4
noise                                                56    14
partnership or collaboration                         31    10
price movement or market trading                     54    13


## 10 · Classifier on new split

In [ ]:
from sklearn.linear_model import LogisticRegression

train = pd.read_csv(f"{DIR}/train.csv")
test = pd.read_csv(f"{DIR}/test.csv")

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(model.encode(train.headline.tolist()), train.label)

probs = clf.predict_proba(model.encode(test.headline.tolist()))
ranked = probs.argsort(axis=1)[:, ::-1]
test["first"] = clf.classes_[ranked[:, 0]]
test["second"] = clf.classes_[ranked[:, 1]]

top1 = (test["first"] == test.label).mean()
top2 = ((test["first"] == test.label) | (test["second"] == test.label)).mean()
print(f"Classifier on new test ({len(test)} headlines) — Top-1: {top1:.0%}   Top-2: {top2:.0%}\n")

print("Per label (top-1):")
print(test.assign(ok=test["first"] == test.label).groupby("label").ok.agg(["sum", "count"]))

test.to_csv(f"{DIR}/test_preds_classifier.csv", index=False)

Classifier on new test (93 headlines) — Top-1: 55%   Top-2: 70%

Per label (top-1):
                                                  sum  count
label                                                       
acquisition or merger                               2      2
funding or investment                               5      8
government regulation, taxation, or legal action    4     10
hack, exploit or security breach                    7      7
institutional adoption or major industry shift      3      8
new product or feature launch                       4     17
new token or coin launch                            3      4
noise                                               4     14
partnership or collaboration                        8     10
price movement or market trading                   11     13


In [ ]:
wrong = test[test["first"] != test.label]
print(pd.crosstab(wrong.label, wrong["first"]))

first                                             acquisition or merger  \
label                                                                     
funding or investment                                                 0   
government regulation, taxation, or legal action                      1   
institutional adoption or major industry shift                        0   
new product or feature launch                                         0   
new token or coin launch                                              0   
noise                                                                 0   
partnership or collaboration                                          0   
price movement or market trading                                      0   

first                                             funding or investment  \
label                                                                     
funding or investment                                                 0   
government regulation, t

## 11 · Claude on new split

In [ ]:
!pip install -q anthropic
import anthropic, json
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-5-5"

NOISE_TEXT = "not a crypto news event (opinion, analysis, explainer, promotion, roundup, or non-crypto news)"
names = sorted(train.label.unique())
shown = [NOISE_TEXT if l == "noise" else l for l in names]

system = "You classify crypto news headlines into exactly these categories:\n" + \
    "\n".join(f"- {l}" for l in shown) + \
    '\n\nReply with JSON only, no other text:\n{"first": "<best label>", "second": "<second-best label>"}\n' + \
    "Use the label text exactly as written above. first and second must be different."

def classify(headline):
    r = client.messages.create(model=MODEL, max_tokens=1024, system=system,
                               messages=[{"role": "user", "content": headline}])
    texts = [b.text for b in r.content if b.type == "text"]
    if not texts:
        return None, None
    text = texts[0].strip().removeprefix("```json").removesuffix("```").strip()
    try:
        out = json.loads(text)
        back = lambda x: "noise" if x == NOISE_TEXT else x
        return back(out.get("first")), back(out.get("second"))
    except json.JSONDecodeError:
        return None, None

claude = test[["headline", "label"]].copy()
claude[["first", "second"]] = [classify(h) for h in claude.headline]

top1 = (claude["first"] == claude.label).mean()
top2 = ((claude["first"] == claude.label) | (claude["second"] == claude.label)).mean()
invalid = (~claude["first"].isin(names)).sum()
print(f"Claude on new test — Top-1: {top1:.0%}   Top-2: {top2:.0%}   Invalid: {invalid}\n")
print(claude.assign(ok=claude["first"] == claude.label).groupby("label").ok.agg(["sum", "count"]))

claude.to_csv(f"{DIR}/test_preds_claude.csv", index=False)

Claude on new test — Top-1: 77%   Top-2: 86%   Invalid: 0

                                                  sum  count
label                                                       
acquisition or merger                               2      2
funding or investment                               8      8
government regulation, taxation, or legal action    8     10
hack, exploit or security breach                    7      7
institutional adoption or major industry shift      5      8
new product or feature launch                      11     17
new token or coin launch                            4      4
noise                                               7     14
partnership or collaboration                       10     10
price movement or market trading                   10     13


## 12 · BART on new split

In [ ]:
from transformers import pipeline

bart = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

rows = []
for h in test.headline:
    r = bart(h, candidate_labels=shown)
    first, second = [("noise" if l == NOISE_TEXT else l) for l in r["labels"][:2]]
    rows.append((first, second))

bart_preds = test[["headline", "label"]].copy()
bart_preds[["first", "second"]] = rows

top1 = (bart_preds["first"] == bart_preds.label).mean()
top2 = ((bart_preds["first"] == bart_preds.label) | (bart_preds["second"] == bart_preds.label)).mean()
print(f"BART on new test — Top-1: {top1:.0%}   Top-2: {top2:.0%}\n")
print(bart_preds.assign(ok=bart_preds["first"] == bart_preds.label).groupby("label").ok.agg(["sum", "count"]))

bart_preds.to_csv(f"{DIR}/test_preds_bart.csv", index=False)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

BART on new test — Top-1: 45%   Top-2: 58%

                                                  sum  count
label                                                       
acquisition or merger                               1      2
funding or investment                               8      8
government regulation, taxation, or legal action    2     10
hack, exploit or security breach                    7      7
institutional adoption or major industry shift      3      8
new product or feature launch                       5     17
new token or coin launch                            1      4
noise                                               0     14
partnership or collaboration                        8     10
price movement or market trading                    7     13


## 13 · Fine-tuning

In [ ]:
!pip install -q transformers datasets accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

train = pd.read_csv(f"{DIR}/train.csv")
test = pd.read_csv(f"{DIR}/test.csv")
print(f"Train: {len(train)}   Test: {len(test)}")

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

example = train.headline[0]
tokens = tokenizer.tokenize(example)
print(f"\n{example}\n→ {len(tokens)} tokens:\n{tokens}")

GPU: Tesla T4
Train: 338   Test: 93


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


The White Whale up 10x in a week! Lighter TGE! Saylor buys $109M BTC!
→ 23 tokens:
['The', 'ĠWhite', 'ĠWhale', 'Ġup', 'Ġ10', 'x', 'Ġin', 'Ġa', 'Ġweek', '!', 'ĠL', 'ighter', 'ĠT', 'GE', '!', 'ĠS', 'aylor', 'Ġbuys', 'Ġ$', '109', 'M', 'ĠBTC', '!']


In [ ]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification

names = sorted(train.label.unique())
label2id = {l: i for i, l in enumerate(names)}
id2label = {i: l for l, i in label2id.items()}

def to_dataset(df):
    ds = Dataset.from_dict({"text": df.headline.tolist(),
                            "label": [label2id[l] for l in df.label]})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=64), batched=True)

train_ds = to_dataset(train)
test_ds = to_dataset(test)

model_ft = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=len(names), id2label=id2label, label2id=label2id)

print(train_ds)
print(f"\nLongest headline: {max(len(x) for x in train_ds['input_ids'])} tokens")

Map:   0%|          | 0/338 [00:00<?, ? examples/s]

Map:   0%|          | 0/93 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 338
})

Longest headline: 41 tokens


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

args = TrainingArguments(
    output_dir="/content/ft-out",
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    weight_decay=0.01,
    warmup_steps=22,
    fp16=True,
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = Trainer(model=model_ft, args=args, train_dataset=train_ds,
                  data_collator=DataCollatorWithPadding(tokenizer))
trainer.train()

Step,Training Loss
22,2.300459
44,2.111549
66,1.615721
88,1.144106
110,0.828090
132,0.649913
154,0.518620
176,0.382598
198,0.309689
220,0.267066


TrainOutput(global_step=220, training_loss=1.012781186537309, metrics={'train_runtime': 18.9992, 'train_samples_per_second': 177.902, 'train_steps_per_second': 11.579, 'total_flos': 49178577318864.0, 'train_loss': 1.012781186537309, 'epoch': 10.0})

In [ ]:
pred = trainer.predict(test_ds)
ranked = pred.predictions.argsort(axis=1)[:, ::-1]

ft = test[["headline", "label"]].copy()
ft["first"] = [id2label[i] for i in ranked[:, 0]]
ft["second"] = [id2label[i] for i in ranked[:, 1]]

top1 = (ft["first"] == ft.label).mean()
top2 = ((ft["first"] == ft.label) | (ft["second"] == ft.label)).mean()

print("Model             Top-1   Top-2")
print("BART              45%     58%")
print("Classifier        55%     70%")
print(f"Fine-tuned        {top1:.0%}     {top2:.0%}")
print("Claude            77%     86%\n")

print(ft.assign(ok=ft["first"] == ft.label).groupby("label").ok.agg(["sum", "count"]))
ft.to_csv(f"{DIR}/test_preds_finetuned.csv", index=False)

Model             Top-1   Top-2
BART              45%     58%
Classifier        55%     70%
Fine-tuned        59%     76%
Claude            77%     86%

                                                  sum  count
label                                                       
acquisition or merger                               2      2
funding or investment                               6      8
government regulation, taxation, or legal action    7     10
hack, exploit or security breach                    7      7
institutional adoption or major industry shift      3      8
new product or feature launch                       7     17
new token or coin launch                            4      4
noise                                               3     14
partnership or collaboration                        7     10
price movement or market trading                    9     13


In [ ]:
for s in [1, 2, 3]:
    m = AutoModelForSequenceClassification.from_pretrained(
        "roberta-base", num_labels=len(names), id2label=id2label, label2id=label2id)
    args.seed = s
    t = Trainer(model=m, args=args, train_dataset=train_ds,
                data_collator=DataCollatorWithPadding(tokenizer))
    t.train()
    r = t.predict(test_ds).predictions.argsort(axis=1)[:, ::-1]
    first = [id2label[i] for i in r[:, 0]]
    second = [id2label[i] for i in r[:, 1]]
    t1 = (pd.Series(first) == test.label).mean()
    t2 = ((pd.Series(first) == test.label) | (pd.Series(second) == test.label)).mean()
    print(f"seed {s}: Top-1 {t1:.0%}   Top-2 {t2:.0%}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
22,2.274450
44,2.013682
66,1.560476
88,1.179408
110,0.918196
132,0.695657
154,0.525507
176,0.430222
198,0.357700
220,0.312328


seed 1: Top-1 58%   Top-2 75%


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (2 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
22,2.276747
44,2.099483
66,1.561033
88,1.148602
110,0.864679
132,0.649769
154,0.483370
176,0.377781
198,0.305845
220,0.299533


seed 3: Top-1 57%   Top-2 72%


## 14 · Claude + rulebook

In [ ]:
!pip install -q anthropic
import anthropic, json
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-5-5"

train = pd.read_csv(f"{DIR}/train.csv")
test = pd.read_csv(f"{DIR}/test.csv")
names = sorted(train.label.unique())

GUIDE = """GENERAL RULES
- Label the main event (who did what), not the background. Text after "as", "amid", "after" is usually background.
- Label from the headline alone. Knowing who named entities are is fine; don't add facts the headline doesn't state.
- Test: "What changed in the world?" If nothing (opinion, commentary, someone noticing something) → skip.
- Two events: label the one that leads. If unclear, a completed event beats a planned one. Opinion + event in one headline → label the event.
- Label the topic, not whether it happened: planned, explored, rumored, suspected, denied or delayed events get the same label as the event itself.
- Text after a colon naming a person, outlet or column (": Bitwise", ": Report", ": 9 outlets compared", ": Asia Express") is the source → ignore it.
- "Here's how / what we know" headlines that report a new event → label the event. Explainers that report no event → skip.
- "Regulator says..." reporting on market activity → label the activity. Regulator acting (approves, rejects, proposes, investigates) → 2.
- Same event with the same parts → same label regardless of word order. Related headlines that name different things get labeled by what they name (e.g. objections to a named SEC rule → 2; companies objecting with no rule or regulator named → 8).
- Aggregate statistics: market/capital numbers (flows, totals raised, TVL, market share) → 7. Adoption numbers (surveys, % of companies using crypto) → 8.
- Prediction markets (Polymarket, Kalshi) count as crypto-related.

SKIP
- Opinion, analysis, explainers ("What is...", "Why..."), listicles, analyst forecasts and price targets, speculation about what might happen.
- Executive or official commentary and quotes with no action.
- Roundups, newsletters, podcast/video episode titles (e.g. "Morning Minute", titles with several "!") that cover several unrelated topics. If every item is the same type of event, or the headline names one event → label it.
- Promotional content: presale reviews, "best tokens to buy", "Token Sale Review & Tokenomics Analysis", price/chart pages. No verb saying what happened = probably not news.
- Non-crypto news (AI, general tech, general cybersecurity, macro, non-crypto deals) unless it directly involves crypto.
- Individuals (KOLs, traders) joining/leaving organizations; executive hires, firings, layoffs within one company.
- Crypto political campaign spending (PACs, election ads).
- Project shutdowns; ceremonies (bell ringing); vague positioning ("positions itself as", "aims to become").
- DAO treasury investments in outside assets.

LABEL DEFINITIONS
1 Hack: crypto protocols, exchanges, wallets or users losing funds or being breached. Includes data breaches, tampered hardware wallets, suspected/denied hacks, losses from internal operational failures, exploit challenges, cryptojacking. Hack aftermath also counts: stolen funds moved/laundered/sold, attribution, cause analysis, victim covering losses. Not: hypothetical threats, disclosed vulnerabilities, security fixes, or delays "over security risks" (→ usually 6). Market reactions to a hack → 7; forecasts about its impact → skip.

2 Regulation/legal: laws or rules proposed, consulted on, approved or in force; tax changes; regulator approvals/rejections; license applications and approvals; crypto firms suspending services in, exiting, or being barred from a country; lawsuits, arrests, court rulings, settlements; government investigations and probes; law-enforcement operations (seizures, raids, fund recoveries); bankruptcy/repayment proceedings (Mt. Gox, FTX); lobbying or pushback on a named proposed rule or regulator; investigations into suspected market misconduct (incl. by platforms). Making rules = 2; a government buying/using crypto = 8. Protocol developers changing code rules → 6, not 2.

3 Partnership: two or more organizations working together ("partner", "team up", "joins X and Y to...", "alliance forms", "sign deal"), incl. partnership talks and broad multi-service business relationships. An executive joining another organization's body counts. Crypto firm + traditional institution: if the deal between them is the news → 3. "X and Y launch [product]" or "X launches [product] with Y as launch partner" → 6.

4 Acquisition: one company buying, merging with or taking control of another, incl. planned, explored, rumored or denied deals.

5 Token launch: a new crypto asset or an event about it: TGE, token sale, airdrop, tokenomics announced, exchange listing of a new/upcoming token (incl. pre-market perps), launch delays. Not: traders or funds selling existing tokens (→ 7).

6 Product/feature: a product, tool, service or feature launched or previewed; an existing product expanding to a new chain, market or client group; a platform adding support for a token, stablecoin or chain ("adds", "integrates", "taps"); network upgrades, hard forks, testnets, protocol rule changes and upgrade bug fixes, incl. delays; governance proposals that change how a protocol works or is funded; a traditional institution launching a specific crypto product. Not: a vulnerability being discovered or disclosed.

7 Price/market: price moves, markets up or down, aggregate flows (ETF flows), on-chain and market metrics (TVL, staking, yields, hashrate, stablecoin supply/share), liquidations, named traders' or funds' trades and whale moves, routine fund rebalancing, crypto company earnings, revenue and company-issued guidance.

8 Institutional adoption/industry shift: companies, governments or central banks considering, buying, holding or using crypto or blockchain; treasury purchases, pauses and companies raising money to buy crypto; first-time entry or exit; institutions restricting, rejecting or objecting to crypto products (when no rule/regulator is named); adoption surveys; credit ratings of crypto assets by rating agencies; major strategic shifts (e.g. a16z dropping "crypto" from its name).

9 Funding: money invested into a crypto company or protocol: VC rounds, strategic investments, ecosystem grants, OTC/strategic token sales to investors, crypto company IPOs (incl. planned). If the investor takes control → 4."""

system = f"""You classify crypto news headlines using the labeling guide below.

{GUIDE}

Headlines the SKIP section covers get the label "noise".
Allowed labels: {", ".join(names)}

Reply with JSON only, no other text:
{{"first": "<best label>", "second": "<second-best label>"}}
Use the label text exactly as listed. first and second must be different."""

def classify(headline):
    r = client.messages.create(model=MODEL, max_tokens=2048, system=system,
                               messages=[{"role": "user", "content": headline}])
    texts = [b.text for b in r.content if b.type == "text"]
    if not texts:
        return None, None
    text = texts[0].strip().removeprefix("```json").removesuffix("```").strip()
    try:
        out = json.loads(text)
        return out.get("first"), out.get("second")
    except json.JSONDecodeError:
        return None, None

cg = test[["headline", "label"]].copy()
cg[["first", "second"]] = [classify(h) for h in cg.headline]
cg.to_csv(f"{DIR}/test_preds_claude_guide.csv", index=False)

top1 = (cg["first"] == cg.label).mean()
top2 = ((cg["first"] == cg.label) | (cg["second"] == cg.label)).mean()
print(f"Claude + guide — Top-1: {top1:.0%}   Top-2: {top2:.0%}   Invalid: {(~cg['first'].isin(names)).sum()}")
print("Claude (no guide): 77% / 86%\n")
print(cg.assign(ok=cg["first"] == cg.label).groupby("label").ok.agg(["sum", "count"]))

Claude + guide — Top-1: 81%   Top-2: 90%   Invalid: 0
Claude (no guide): 77% / 86%

                                                  sum  count
label                                                       
acquisition or merger                               1      2
funding or investment                               8      8
government regulation, taxation, or legal action    6     10
hack, exploit or security breach                    7      7
institutional adoption or major industry shift      4      8
new product or feature launch                      15     17
new token or coin launch                            4      4
noise                                              11     14
partnership or collaboration                        8     10
price movement or market trading                   11     13


In [ ]:
wrong = cg[cg["first"] != cg.label]
for _, r in wrong.iterrows():
    print(f"you: {r.label}\nclaude: {r['first']}\n  {r.headline}\n")

you: institutional adoption or major industry shift
claude: price movement or market trading
  Ethereum Foundation “dump” is a $6.4m treasury sale, not a crisis

you: institutional adoption or major industry shift
claude: noise
  Circle faces up to 20% revenue risk from AI agents, Mizuho says

you: acquisition or merger
claude: noise
  Ares Capital Corporation Agrees to Acquire MidCap Financial Investment Corporation

you: price movement or market trading
claude: noise
  Bitcoin price analysis sees ‘strong rally’ as old coins become dormant

you: new product or feature launch
claude: partnership or collaboration
  AmericanFortress, Sumsub bring verified ID to crypto

you: partnership or collaboration
claude: new product or feature launch
  Robinhood taps Ripple’s $2.5 billion RLUSD stablecoin despite backing rival USDG

you: institutional adoption or major industry shift
claude: new product or feature launch
  Nasdaq-100 comes onchain as Janus Henderson and Centrifuge launch first lice

In [ ]:
fixes = {
    "Ethereum Foundation": "price movement or market trading",
    "Circle faces up to 20%": "noise",
    "Ares Capital": "noise",
    "Bitcoin price analysis sees": "noise",
    "Nasdaq-100 comes onchain": "new product or feature launch",
    "XRP News: Ripple Partners": "partnership or collaboration",
    "Bessent sees crypto": "noise",
    "Zumo and Quantum Chain": "partnership or collaboration",
    "Comer Probes Delaware": "noise",
}

test = pd.read_csv(f"{DIR}/test.csv")
test2 = test.copy()
for text, lab in fixes.items():
    hit = test2.headline.str.contains(text, regex=False)
    print(f"{hit.sum()}× {text}")
    test2.loc[hit, "label"] = lab
test2.to_csv(f"{DIR}/test_v2.csv", index=False)

# same fixes in labeled.csv ("skip" there, not "noise")
done = pd.read_csv(LABELED)
for text, lab in fixes.items():
    done.loc[done.headline.str.contains(text, regex=False), "label"] = "skip" if lab == "noise" else lab
done.to_csv(LABELED, index=False)

key = dict(zip(test2.headline, test2.label))
print("\nModel             Top-1 (v1 → v2)   Top-2 (v1 → v2)")
for name, f in [("BART", "bart"), ("Classifier", "classifier"), ("Fine-tuned", "finetuned"),
                ("Claude", "claude"), ("Claude + guide", "claude_guide")]:
    p = pd.read_csv(f"{DIR}/test_preds_{f}.csv")
    new = p.headline.map(key)
    t1a, t1b = (p["first"] == p.label).mean(), (p["first"] == new).mean()
    t2a = ((p["first"] == p.label) | (p["second"] == p.label)).mean()
    t2b = ((p["first"] == new) | (p["second"] == new)).mean()
    print(f"{name:<17} {t1a:.0%} → {t1b:.0%}         {t2a:.0%} → {t2b:.0%}")

1× Ethereum Foundation
1× Circle faces up to 20%
1× Ares Capital
1× Bitcoin price analysis sees
1× Nasdaq-100 comes onchain
1× XRP News: Ripple Partners
1× Bessent sees crypto
1× Zumo and Quantum Chain
1× Comer Probes Delaware

Model             Top-1 (v1 → v2)   Top-2 (v1 → v2)
BART              45% → 45%         58% → 57%
Classifier        55% → 57%         70% → 71%
Fine-tuned        59% → 58%         76% → 77%
Claude            77% → 78%         86% → 90%
Claude + guide    81% → 90%         90% → 97%


## 15 · Blind review

In [ ]:
test = pd.read_csv(f"{DIR}/test.csv")
for i, r in test.iterrows():
    print(f"{i:>2}  [{r.label}]  {r.headline}")

 0  [government regulation, taxation, or legal action]  Ripple gets a final go-ahead to launch its own U.S. national trust bank
 1  [partnership or collaboration]  BlackRock joins SBI and Startale to link tokenized MMFs with Japanese yen stablecoins
 2  [hack, exploit or security breach]  Keystone breach deepens: Hackers stole key used to verify wallets as genuine
 3  [government regulation, taxation, or legal action]  Ripple gains final OCC approval to launch national trust bank and bring RLUSD under federal oversight
 4  [institutional adoption or major industry shift]  Ethereum Foundation “dump” is a $6.4m treasury sale, not a crisis
 5  [noise]  StoneX lifts Strive price target 34% as new stock offering bolsters bitcoin strategy
 6  [funding or investment]  RedotPay weighs confidential US IPO filing by year-end: Bloomberg
 7  [new product or feature launch]  Zcash co-founder proposes ending grants funding, citing $125 million treasury
 8  [institutional adoption or major industry s

In [ ]:
changes = {4: "price movement or market trading", 8: "noise",
           29: "new product or feature launch", 36: "partnership or collaboration",
           53: "noise", 71: "partnership or collaboration", 88: "noise"}

test = pd.read_csv(f"{DIR}/test.csv")
test3 = test.copy()
for i, lab in changes.items():
    test3.loc[i, "label"] = lab
test3.to_csv(f"{DIR}/test_v3.csv", index=False)

done = pd.read_csv(LABELED)
for i, lab in changes.items():
    done.loc[done.headline == test.headline[i], "label"] = "skip" if lab == "noise" else lab
done.to_csv(LABELED, index=False)

key = dict(zip(test3.headline, test3.label))
print("Model             Top-1 (orig → blind)   Top-2 (orig → blind)")
for name, f in [("BART", "bart"), ("Classifier", "classifier"), ("Fine-tuned", "finetuned"),
                ("Claude", "claude"), ("Claude + guide", "claude_guide")]:
    p = pd.read_csv(f"{DIR}/test_preds_{f}.csv")
    new = p.headline.map(key)
    a1, b1 = (p["first"] == p.label).mean(), (p["first"] == new).mean()
    a2 = ((p["first"] == p.label) | (p["second"] == p.label)).mean()
    b2 = ((p["first"] == new) | (p["second"] == new)).mean()
    print(f"{name:<17} {a1:.0%} → {b1:.0%}              {a2:.0%} → {b2:.0%}")

Model             Top-1 (orig → blind)   Top-2 (orig → blind)
BART              45% → 47%              58% → 59%
Classifier        55% → 59%              70% → 73%
Fine-tuned        59% → 60%              76% → 78%
Claude            77% → 81%              86% → 90%
Claude + guide    81% → 88%              90% → 97%


## 16 · Rebuild

In [5]:
is_binance_hack = (
    news.headline.str.contains("Binance", case=False)
    & news.headline.str.contains("hack|heist|breach|stolen|exploit|attacker|hacker|loot|SAFU|theft",
                                 case=False, regex=True)
)
bh = news[is_binance_hack]
print(f"{len(bh)} headlines in {bh.story.nunique()} story groups\n")
print(bh.label.value_counts(), "\n")
for _, r in bh.sort_values("label").iterrows():
    print(f"[{r.label[:12]}] {r.headline}")

88 headlines in 33 story groups

label
hack, exploit or security breach                    72
price movement or market trading                    12
government regulation, taxation, or legal action     2
institutional adoption or major industry shift       2
Name: count, dtype: int64 

[government r] South Korea urges exchanges to boost cold storage after Binance hack
[government r] Fed, OCC check US banks after Binance hack as experts flag security risks
[hack, exploi] Binance Hacker Dumps FUN: Token Crashes 90% [2026]
[hack, exploi] The Binance Hacker Sold $40 Million of Uniswap and $34 Million of Ethena. Which Recovers First?
[hack, exploi] President Trump says they’re looking into Binance hack as calls grow for crypto rules
[hack, exploi] Inside the largest crypto hack on record: How attackers breached Binance for $6.3 billion
[hack, exploi] North Korean hackers launder $242 million of Binance loot via Tron as FBI presses THORChain: analyst
[hack, exploi] Binance completes latest p

In [6]:
fixes = {
    "Biggest Crypto Hacks in History: Is Binance No. 1": "skip",
    "Standard Chartered sees more pain after": "skip",
    "Binance hack sends bitcoin, ether tumbling": "price movement or market trading",
    "Crypto liquidations jump to $1.57 billion": "price movement or market trading",
    "Secret Service, FBI Join Binance Hack Probe": "government regulation, taxation, or legal action",
}
done = pd.read_csv(LABELED)
for text, lab in fixes.items():
    hit = done.headline.str.contains(text, regex=False)
    print(f"{hit.sum()}× {text}")
    done.loc[hit, "label"] = lab
done.to_csv(LABELED, index=False)

1× Biggest Crypto Hacks in History: Is Binance No. 1
1× Standard Chartered sees more pain after
1× Binance hack sends bitcoin, ether tumbling
1× Crypto liquidations jump to $1.57 billion
1× Secret Service, FBI Join Binance Hack Probe


In [7]:
is_bh = (
    news.headline.str.contains("Binance", case=False)
    & news.headline.str.contains("hack|heist|breach|stolen|exploit|attacker|hacker|loot|SAFU|theft",
                                 case=False, regex=True)
    & ~news.headline.str.contains("Mystery \\$106 Million|Bitmine", regex=True)
)
news["event"] = news.story.astype(str)
news.loc[is_bh, "event"] = "binance_hack"

kept = news[news.keep]
bh_kept = kept[kept.event == "binance_hack"]
print(f"Binance event: {is_bh.sum()} headlines, {len(bh_kept)} distinct stories")
print(bh_kept.label.value_counts())

Binance event: 85 headlines, 31 distinct stories
label
hack, exploit or security breach                    24
price movement or market trading                     5
government regulation, taxation, or legal action     2
Name: count, dtype: int64


In [11]:
done = pd.read_csv(LABELED)
pool = pd.read_csv(POOL)
leaks = pd.read_csv(f"{DIR}/test21_overlap.csv")

done = done.merge(pool[["headline", "collected"]], on="headline", how="left")
done["date"] = pd.to_datetime(done.published, utc=True, errors="coerce", format="mixed")
done["date"] = done.date.fillna(pd.to_datetime(done.collected, utc=True))

# news: one headline per story, dated by its EVENT's earliest headline
nd = news.merge(done[["headline", "date"]], on="headline", how="left")
nd = nd[~nd.headline.isin(leaks.headline)]
nd["date"] = nd.groupby("event").date.transform("min")
one = nd[~nd.story.duplicated()][["headline", "label", "date", "event"]]

# cap Binance hack stories at 5
bh_hack = one[(one.event == "binance_hack") & (one.label == "hack, exploit or security breach")]
one = one.drop(bh_hack.index.difference(bh_hack.sample(n=5, random_state=42).index))

# noise: skips, capped near the largest class
skips = done[done.label == "skip"]
noise = skips.sample(n=min(140, len(skips)), random_state=42)[["headline", "date"]]
noise["label"] = "noise"
noise["event"] = "noise_" + noise.index.astype(str)

dataset2 = pd.concat([one, noise]).sort_values("date").reset_index(drop=True)
dataset2.to_csv(f"{DIR}/dataset_v2.csv", index=False)

print(f"{len(dataset2)} examples\n")
print(dataset2.label.value_counts())
days = dataset2.date.dt.date.value_counts().sort_index()
print("\nBefore Oct 20:", days[days.index < pd.Timestamp("2026-10-20").date()].sum())
print(days[days.index >= pd.Timestamp("2026-10-20").date()])

800 examples

label
price movement or market trading                    143
noise                                               140
government regulation, taxation, or legal action    109
new product or feature launch                       101
hack, exploit or security breach                     86
institutional adoption or major industry shift       68
partnership or collaboration                         55
funding or investment                                52
new token or coin launch                             24
acquisition or merger                                22
Name: count, dtype: int64

Before Oct 20: 58
date
2026-10-20    32
2026-10-21    65
2026-10-22    65
2026-10-23    50
2026-10-24    21
2026-10-25     9
2026-10-26    55
2026-10-27    52
2026-10-28    53
2026-10-29    71
2026-10-30    27
2026-10-31     9
2026-11-01     2
2026-11-02    46
2026-11-03    46
2026-11-04    48
2026-11-05    70
2026-11-06    21
Name: count, dtype: int64


In [12]:
CUTOFF = pd.Timestamp("2026-11-03", tz="UTC")

dataset2 = pd.read_csv(f"{DIR}/dataset_v2.csv", parse_dates=["date"])
train_r2 = dataset2[dataset2.date < CUTOFF].reset_index(drop=True)
test_r2 = dataset2[dataset2.date >= CUTOFF].reset_index(drop=True)

assert not set(train_r2.event) & set(test_r2.event), "an event is on both sides!"

train_r2.to_csv(f"{DIR}/train_r2.csv", index=False)
test_r2.to_csv(f"{DIR}/test_r2.csv", index=False)

print(f"Train: {len(train_r2)}   Test: {len(test_r2)}\n")
print(pd.DataFrame({"train": train_r2.label.value_counts(),
                    "test": test_r2.label.value_counts()}).fillna(0).astype(int))

Train: 615   Test: 185

                                                  train  test
label                                                        
acquisition or merger                                20     2
funding or investment                                43     9
government regulation, taxation, or legal action     76    33
hack, exploit or security breach                     65    21
institutional adoption or major industry shift       59     9
new product or feature launch                        73    28
new token or coin launch                             18     6
noise                                               111    29
partnership or collaboration                         49     6
price movement or market trading                    101    42


## 17 · Round 2 models

In [5]:
from sklearn.linear_model import LogisticRegression

train_r2 = pd.read_csv(f"{DIR}/train_r2.csv")
test_r2 = pd.read_csv(f"{DIR}/test_r2.csv")

def score(preds, name):
    t1 = (preds["first"] == preds.label).mean()
    t2 = ((preds["first"] == preds.label) | (preds["second"] == preds.label)).mean()
    print(f"{name:<20} Top-1 {t1:.0%}   Top-2 {t2:.0%}")

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(model.encode(train_r2.headline.tolist()), train_r2.label)
ranked = clf.predict_proba(model.encode(test_r2.headline.tolist())).argsort(axis=1)[:, ::-1]

p_clf = test_r2[["headline", "label"]].copy()
p_clf["first"] = clf.classes_[ranked[:, 0]]
p_clf["second"] = clf.classes_[ranked[:, 1]]
p_clf.to_csv(f"{DIR}/r2_preds_classifier.csv", index=False)

score(p_clf, "Classifier")
print("Round 1 (93 test, 338 train): 59% after blind review")

Classifier           Top-1 68%   Top-2 79%
Round 1 (93 test, 338 train): 59% after blind review


In [6]:
!pip install -q anthropic
import anthropic, json
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-5-5"
names = sorted(train_r2.label.unique())

GUIDE = """GENERAL RULES
- Label the main event (who did what), not the background. Text after "as", "amid", "after" is usually background.
- Label from the headline alone. Knowing who named entities are is fine; don't add facts the headline doesn't state.
- Test: "What changed in the world?" If nothing (opinion, commentary, someone noticing something) → skip.
- Two events: label the one that leads. If unclear, a completed event beats a planned one. Opinion + event in one headline → label the event.
- Label the topic, not whether it happened: planned, explored, rumored, suspected, denied or delayed events get the same label as the event itself.
- Text after a colon naming a person, outlet or column (": Bitwise", ": Report", ": 9 outlets compared", ": Asia Express") is the source → ignore it.
- "Here's how / what we know" headlines that report a new event → label the event. Explainers that report no event → skip.
- "Regulator says..." reporting on market activity → label the activity. Regulator acting (approves, rejects, proposes, investigates) → 2.
- Same event with the same parts → same label regardless of word order. Related headlines that name different things get labeled by what they name (e.g. objections to a named SEC rule → 2; companies objecting with no rule or regulator named → 8).
- Aggregate statistics: market/capital numbers (flows, totals raised, TVL, market share) → 7. Adoption numbers (surveys, % of companies using crypto) → 8.
- Prediction markets (Polymarket, Kalshi) count as crypto-related.

SKIP
- Opinion, analysis, explainers ("What is...", "Why..."), listicles, analyst forecasts and price targets, speculation about what might happen.
- Executive or official commentary and quotes with no action.
- Roundups, newsletters, podcast/video episode titles (e.g. "Morning Minute", titles with several "!") that cover several unrelated topics. If every item is the same type of event, or the headline names one event → label it.
- Promotional content: presale reviews, "best tokens to buy", "Token Sale Review & Tokenomics Analysis", price/chart pages. No verb saying what happened = probably not news.
- Non-crypto news (AI, general tech, general cybersecurity, macro, non-crypto deals) unless it directly involves crypto.
- Individuals (KOLs, traders) joining/leaving organizations; executive hires, firings, layoffs within one company.
- Crypto political campaign spending (PACs, election ads).
- Project shutdowns; ceremonies (bell ringing); vague positioning ("positions itself as", "aims to become").
- DAO treasury investments in outside assets.

LABEL DEFINITIONS
1 Hack: crypto protocols, exchanges, wallets or users losing funds or being breached. Includes data breaches, tampered hardware wallets, suspected/denied hacks, losses from internal operational failures, exploit challenges, cryptojacking. Hack aftermath also counts: stolen funds moved/laundered/sold, attribution, cause analysis, victim covering losses. Not: hypothetical threats, disclosed vulnerabilities, security fixes, or delays "over security risks" (→ usually 6). Market reactions to a hack → 7; forecasts about its impact → skip.

2 Regulation/legal: laws or rules proposed, consulted on, approved or in force; tax changes; regulator approvals/rejections; license applications and approvals; crypto firms suspending services in, exiting, or being barred from a country; lawsuits, arrests, court rulings, settlements; government investigations and probes; law-enforcement operations (seizures, raids, fund recoveries); bankruptcy/repayment proceedings (Mt. Gox, FTX); lobbying or pushback on a named proposed rule or regulator; investigations into suspected market misconduct (incl. by platforms). Making rules = 2; a government buying/using crypto = 8. Protocol developers changing code rules → 6, not 2.

3 Partnership: two or more organizations working together ("partner", "team up", "joins X and Y to...", "alliance forms", "sign deal"), incl. partnership talks and broad multi-service business relationships. An executive joining another organization's body counts. Crypto firm + traditional institution: if the deal between them is the news → 3. "X and Y launch [product]" or "X launches [product] with Y as launch partner" → 6.

4 Acquisition: one company buying, merging with or taking control of another, incl. planned, explored, rumored or denied deals.

5 Token launch: a new crypto asset or an event about it: TGE, token sale, airdrop, tokenomics announced, exchange listing of a new/upcoming token (incl. pre-market perps), launch delays. Not: traders or funds selling existing tokens (→ 7).

6 Product/feature: a product, tool, service or feature launched or previewed; an existing product expanding to a new chain, market or client group; a platform adding support for a token, stablecoin or chain ("adds", "integrates", "taps"); network upgrades, hard forks, testnets, protocol rule changes and upgrade bug fixes, incl. delays; governance proposals that change how a protocol works or is funded; a traditional institution launching a specific crypto product. Not: a vulnerability being discovered or disclosed.

7 Price/market: price moves, markets up or down, aggregate flows (ETF flows), on-chain and market metrics (TVL, staking, yields, hashrate, stablecoin supply/share), liquidations, named traders' or funds' trades and whale moves, routine fund rebalancing, crypto company earnings, revenue and company-issued guidance.

8 Institutional adoption/industry shift: companies, governments or central banks considering, buying, holding or using crypto or blockchain; treasury purchases, pauses and companies raising money to buy crypto; first-time entry or exit; institutions restricting, rejecting or objecting to crypto products (when no rule/regulator is named); adoption surveys; credit ratings of crypto assets by rating agencies; major strategic shifts (e.g. a16z dropping "crypto" from its name).

9 Funding: money invested into a crypto company or protocol: VC rounds, strategic investments, ecosystem grants, OTC/strategic token sales to investors, crypto company IPOs (incl. planned). If the investor takes control → 4."""

fmt = f'''Allowed labels: {", ".join(names)}

Reply with JSON only, no other text:
{{"first": "<best label>", "second": "<second-best label>"}}
Use the label text exactly as listed. first and second must be different.'''

sys_plain = ("You classify crypto news headlines. \"noise\" means not a crypto news event "
             "(opinion, analysis, explainer, promotion, roundup, or non-crypto news).\n\n" + fmt)
sys_guide = (f"You classify crypto news headlines using the labeling guide below.\n\n{GUIDE}\n\n"
             "Headlines the SKIP section covers get the label \"noise\".\n" + fmt)

def ask(headline, system):
    r = client.messages.create(model=MODEL, max_tokens=2048, system=system,
                               messages=[{"role": "user", "content": headline}])
    texts = [b.text for b in r.content if b.type == "text"]
    try:
        out = json.loads(texts[0].strip().removeprefix("```json").removesuffix("```").strip())
        return out.get("first"), out.get("second")
    except (IndexError, json.JSONDecodeError):
        return None, None

for name, system, f in [("Claude", sys_plain, "claude"), ("Claude + guide", sys_guide, "claude_guide")]:
    p = test_r2[["headline", "label"]].copy()
    p[["first", "second"]] = [ask(h, system) for h in p.headline]
    p.to_csv(f"{DIR}/r2_preds_{f}.csv", index=False)
    score(p, name)

print("\nRound 1 (blind-reviewed): Claude 81% · Claude + guide 88%")

Claude               Top-1 68%   Top-2 84%
Claude + guide       Top-1 79%   Top-2 91%

Round 1 (blind-reviewed): Claude 81% · Claude + guide 88%


In [7]:
!pip install -q transformers datasets accelerate sentencepiece
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

train_r2 = pd.read_csv(f"{DIR}/train_r2.csv")
test_r2 = pd.read_csv(f"{DIR}/test_r2.csv")
names = sorted(train_r2.label.unique())
label2id = {l: i for i, l in enumerate(names)}
id2label = {i: l for l, i in label2id.items()}

for base, f in [("roberta-base", "roberta"), ("microsoft/deberta-v3-base", "deberta")]:
    tok = AutoTokenizer.from_pretrained(base)

    def to_ds(df):
        ds = Dataset.from_dict({"text": df.headline.tolist(),
                                "label": [label2id[l] for l in df.label]})
        return ds.map(lambda b: tok(b["text"], truncation=True, max_length=64), batched=True)

    m = AutoModelForSequenceClassification.from_pretrained(
        base, num_labels=len(names), id2label=id2label, label2id=label2id)

    args = TrainingArguments(
        output_dir=f"/content/ft-{f}", num_train_epochs=10, learning_rate=2e-5,
        per_device_train_batch_size=16, weight_decay=0.01, warmup_steps=39,
        fp16=(f == "roberta"), logging_strategy="epoch", save_strategy="no",
        report_to="none", seed=42)

    t = Trainer(model=m, args=args, train_dataset=to_ds(train_r2),
                data_collator=DataCollatorWithPadding(tok))
    t.train()

    ranked = t.predict(to_ds(test_r2)).predictions.argsort(axis=1)[:, ::-1]
    p = test_r2[["headline", "label"]].copy()
    p["first"] = [id2label[i] for i in ranked[:, 0]]
    p["second"] = [id2label[i] for i in ranked[:, 1]]
    p.to_csv(f"{DIR}/r2_preds_{f}.csv", index=False)
    score(p, base)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

... (17 outputs left out)

Map:   0%|          | 0/615 [00:00<?, ? examples/s]

Step,Training Loss
39,2.228837
78,2.212235
117,0.000000
156,0.000000
195,0.000000
234,0.000000
273,0.000000
312,0.000000
351,0.000000
390,0.000000


Map:   0%|          | 0/185 [00:00<?, ? examples/s]

microsoft/deberta-v3-base Top-1 23%   Top-2 26%


In [8]:
base, f = "microsoft/deberta-v3-base", "deberta"
tok = AutoTokenizer.from_pretrained(base)

def to_ds(df):
    ds = Dataset.from_dict({"text": df.headline.tolist(),
                            "label": [label2id[l] for l in df.label]})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=64), batched=True)

m = AutoModelForSequenceClassification.from_pretrained(
    base, num_labels=len(names), id2label=id2label, label2id=label2id)
args = TrainingArguments(
    output_dir="/content/ft-deberta", num_train_epochs=10, learning_rate=1e-5,
    per_device_train_batch_size=16, weight_decay=0.01, warmup_steps=39,
    fp16=False, logging_strategy="epoch", save_strategy="no", report_to="none", seed=42)
t = Trainer(model=m, args=args, train_dataset=to_ds(train_r2),
            data_collator=DataCollatorWithPadding(tok))
t.train()

ranked = t.predict(to_ds(test_r2)).predictions.argsort(axis=1)[:, ::-1]
p = test_r2[["headline", "label"]].copy()
p["first"] = [id2label[i] for i in ranked[:, 0]]
p["second"] = [id2label[i] for i in ranked[:, 1]]
p.to_csv(f"{DIR}/r2_preds_deberta.csv", index=False)
score(p, base)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

Map:   0%|          | 0/615 [00:00<?, ? examples/s]

Step,Training Loss
39,2.226175
78,2.295461
117,2.042462
156,0.000000
195,0.000000
234,0.000000
273,0.000000
312,0.000000
351,0.000000
390,0.000000


Map:   0%|          | 0/185 [00:00<?, ? examples/s]

microsoft/deberta-v3-base Top-1 23%   Top-2 26%


In [9]:
base = "roberta-base"
tok = AutoTokenizer.from_pretrained(base)

def to_ds(df):
    ds = Dataset.from_dict({"text": df.headline.tolist(),
                            "label": [label2id[l] for l in df.label]})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=64), batched=True)

for s in [1, 2, 3]:
    m = AutoModelForSequenceClassification.from_pretrained(
        base, num_labels=len(names), id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir="/content/ft-seed", num_train_epochs=10, learning_rate=2e-5,
        per_device_train_batch_size=16, weight_decay=0.01, warmup_steps=39,
        fp16=True, logging_strategy="no", save_strategy="no", report_to="none", seed=s)
    t = Trainer(model=m, args=args, train_dataset=to_ds(train_r2),
                data_collator=DataCollatorWithPadding(tok))
    t.train()
    ranked = t.predict(to_ds(test_r2)).predictions.argsort(axis=1)[:, ::-1]
    p = test_r2[["headline", "label"]].copy()
    p["first"] = [id2label[i] for i in ranked[:, 0]]
    p["second"] = [id2label[i] for i in ranked[:, 1]]
    score(p, f"roberta seed {s}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/615 [00:00<?, ? examples/s]

Step,Training Loss


Map:   0%|          | 0/185 [00:00<?, ? examples/s]

roberta seed 1       Top-1 71%   Top-2 85%


... (8 outputs left out)

Map:   0%|          | 0/615 [00:00<?, ? examples/s]

Step,Training Loss


Map:   0%|          | 0/185 [00:00<?, ? examples/s]

roberta seed 3       Top-1 75%   Top-2 86%
